In [6]:
# This calculator is based on the python code written by Matthew Partridge from Cranfield University licensed under GPL
# This code in turn is based on the Matlab code and online calculator written by Chris Westbrook
# http://www.met.reading.ac.uk/~sws04cdw/viscosity_calc.html


# Packages
import math
import numpy as np
from scipy.optimize import root_scalar

# Variables

T = 25 				 # Temperature (degrees Celcius)
mass_fraction = 0.8  # Mass fraction of glycerol
solution_mass = 0.06 # Mass of the solution (kg)
f_d = 20             # Driving frequency (Hz)
s_manual = 0.065     # Surface tension (overrides one from interpolated table data) (N/m) 

# Gravity and Densities

glycerolDen = (1273.3-0.6121*T)     			    # Density of Glycerol (kg/m3)
waterDen = (1-math.pow(((abs(T-4))/622),1.7))*1000 	# Density of water (kg/m3)
g = 9.81                                            # Acceleration due to gravity (m/s2)

# Fraction calculator

vol_fraction=(mass_fraction/glycerolDen)/(mass_fraction/glycerolDen+(1-mass_fraction)/waterDen)

print("Volume fraction of mixture =",vol_fraction)

# Surface tension (interpolated for 25 C)

s = np.array([0.06973,0.06895,0.06818,0.06769,0.06721,0.06622,0.06517])
if mass_fraction == 0.2:
    s = s[0]
elif mass_fraction == 0.3:
    s = s[1]
elif mass_fraction == 0.4:
    s = s[2]
elif mass_fraction == 0.5:
    s = s[3]
elif mass_fraction == 0.6:
    s = s[4]
elif mass_fraction == 0.7:
    s = s[5]
elif mass_fraction == 0.8:
    s = s[6]
else:
    s = s_manual

# Density calculator

## Andreas Volk polynomial method
contraction_av=1-math.pow(3.520E-8*((mass_fraction*100)),3)+math.pow(1.027E-6*((mass_fraction*100)),2)+2.5E-4*(mass_fraction*100)-1.691E-4
contraction=1+contraction_av/100

rho_mix=(glycerolDen*vol_fraction+waterDen*(1-vol_fraction))*contraction

print("Density of mixture =",rho_mix,"kg/m3")

# Height calculator

h = solution_mass/(rho_mix*0.01)

print("Solution depth =",h)

# Viscosity calculator

glycerolVisc=0.001*12100*np.exp((-1233+T)*T/(9900+70*T))
waterVisc=0.001*1.790*np.exp((-1230-T)*T/(36100+360*T))

a=0.705-0.0017*T
b=(4.9+0.036*T)*np.power(a,2.5)
alpha=1-mass_fraction+(a*b*mass_fraction*(1-mass_fraction))/(a*mass_fraction+b*(1-mass_fraction))
A=np.log(waterVisc/glycerolVisc)

dyn_visc=glycerolVisc*np.exp(A*alpha)
kin_visc=dyn_visc/rho_mix

print("Dynamic viscosity of mixture =",dyn_visc, "Ns/m2")
print("Kinematic viscosity of mixture =",kin_visc, "m2/s")


# Wavenumber (assuming w-2*w_0=0)
f_0=f_d/2                  # Dominant frequency of the waves (Hz)
w_d=2*math.pi*f_d
w_0=2*math.pi*f_0

# Help from ChatGPT was used in generating the numerical calculation of k from w_0
def dispersion_relation(k):
    return ((g*k + (s/rho_mix)*k**3) * np.tanh(k*h)
            - w_0**2)

sol = root_scalar(
    dispersion_relation,
    bracket=[1e-6, 1000],  # Interval containing the root
    method='brentq'
)

k = sol.root
print("Wavenumber =",round(k,5), "rad/m")

# Linear damping coefficient

damp_coeff=kin_visc*k**2*(2+math.cosh(2*k*h)/((math.sinh(2*k*h))**2))+math.sqrt(k*kin_visc*math.sqrt(g*h)/8)*2*k/math.sinh(2*k*h)
damp_coeff_approx=2*kin_visc*k**2

print("Linear damping coefficient =",round(damp_coeff,5), "rad2/s")
print("Approximate linear damping coefficient =",round(damp_coeff_approx,5), "rad2/s")

# On-set acceleration (assuming w-2*w_0=0)

gamma_c=4*damp_coeff*w_0/(g*k*math.tanh(k*h))
gamma_c_approx=4*damp_coeff_approx*w_0/(g*k*math.tanh(k*h))

print("On-set acceleration =",round(gamma_c,5), "a/g")
print("Approximate on-set acceleration =",round(gamma_c_approx,5), "a/g")

Volume fraction of mixture = 0.7601711539734026
Density of mixture = 1207.5574837975128 kg/m3
Solution depth = 0.004968707560928089
Dynamic viscosity of mixture = 0.04535040088846914 Ns/m2
Kinematic viscosity of mixture = 3.755547996427609e-05 m2/s
Wavenumber = 298.94532 rad/m
Linear damping coefficient = 8.14127 rad2/s
Approximate linear damping coefficient = 6.71254 rad2/s
On-set acceleration = 0.77311 a/g
Approximate on-set acceleration = 0.63743 a/g
